In [2]:
import pandas as pd

In [3]:
# Load the datasets
amazon_df = pd.read_csv('./00_raw_data/amazon_sales.csv')
shopify_df = pd.read_csv('./00_raw_data/shopify_sales.csv')
ads_df = pd.read_csv('./00_raw_data/ad_spend.csv')


# Display first few rows
print("Amazon Sales:")
print(amazon_df.head(), "\n")

print("Shopify Sales:")
print(shopify_df.head(), "\n")

print("Ad Spend:")
print(ads_df.head())

Amazon Sales:
  Order_ID        Date         Product     Category  Units_Sold  \
0    A1001  2025-01-01      Phone Case  Accessories          10   
1    A1002  2025-01-02    Laptop Stand       Office          15   
2    A1003  2025-01-03  Wireless Mouse  Electronics          25   
3    A1004  2025-01-04        Keyboard  Electronics          12   
4    A1005  2025-01-05         Charger  Accessories           8   

   Unit_Price_GHS  Revenue_GHS  Refunds      Region  
0              50          500        0       Accra  
1             180         2700        1      Kumasi  
2             120         3000        0    Takoradi  
3             200         2400        0      Tamale  
4              60          480        0  Cape Coast   

Shopify Sales:
  Order_ID        Date   Product     Category  Units_Sold  Unit_Price_GHS  \
0    S2001  2025-01-01    Hoodie      Apparel           5             250   
1    S2002  2025-01-02       Mug         Home          20              70   
2    S2003 

In [4]:
# Check info for each dataset
print("Amazon Info:")
print(amazon_df.info(), "\n")

print("Shopify Info:")
print(shopify_df.info(), "\n")

print("Ad Spend Info:")
print(ads_df.info(), "\n")

# Check for missing values
print("Missing Values:")
print({
    "Amazon": amazon_df.isnull().sum().sum(),
    "Shopify": shopify_df.isnull().sum().sum(),
    "Ads": ads_df.isnull().sum().sum()
})


Amazon Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Order_ID        30 non-null     object
 1   Date            30 non-null     object
 2   Product         30 non-null     object
 3   Category        30 non-null     object
 4   Units_Sold      30 non-null     int64 
 5   Unit_Price_GHS  30 non-null     int64 
 6   Revenue_GHS     30 non-null     int64 
 7   Refunds         30 non-null     int64 
 8   Region          30 non-null     object
dtypes: int64(4), object(5)
memory usage: 2.2+ KB
None 

Shopify Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Order_ID        30 non-null     object
 1   Date            30 non-null     object
 2   Product         30 non-null     object
 3   Cat

In [6]:
# Convert Date columns to datetime
amazon_df["Date"] = pd.to_datetime(amazon_df["Date"])
shopify_df["Date"] = pd.to_datetime(shopify_df["Date"])
ads_df["Date"] = pd.to_datetime(ads_df["Date"])

# Ensure numeric columns are properly typed
numeric_cols_amazon = ["Units_Sold", "Unit_Price_GHS", "Revenue_GHS"]
numeric_cols_shopify = ["Units_Sold", "Unit_Price_GHS", "Revenue_GHS", "Discount_GHS"]
numeric_cols_ads = ["Spend_GHS", "Clicks", "Impressions", "Orders", "Revenue_GHS"]

for col in numeric_cols_amazon:
    amazon_df[col] = pd.to_numeric(amazon_df[col], errors="coerce")

for col in numeric_cols_shopify:
    shopify_df[col] = pd.to_numeric(shopify_df[col], errors="coerce")

for col in numeric_cols_ads:
    ads_df[col] = pd.to_numeric(ads_df[col], errors="coerce")

# Confirm data types
print("\n Data types cleaned successfully!")
print(amazon_df.dtypes)



 Data types cleaned successfully!
Order_ID                  object
Date              datetime64[ns]
Product                   object
Category                  object
Units_Sold                 int64
Unit_Price_GHS             int64
Revenue_GHS                int64
Refunds                    int64
Region                    object
dtype: object


In [7]:
amazon_df.to_csv("./01_clean_data/cleaned_amazon_sales.csv", index=False)
shopify_df.to_csv("./01_clean_data/cleaned_shopify_sales.csv", index=False)
ads_df.to_csv("./01_clean_data/cleaned_ad_spend.csv", index=False)

Data Analysis — computing KPIs like total revenue, top products, ROI

In [8]:
# === TOTAL REVENUE AND UNITS SOLD ===
total_amazon_revenue = amazon_df["Revenue_GHS"].sum()
total_shopify_revenue = shopify_df["Revenue_GHS"].sum()
total_revenue = total_amazon_revenue + total_shopify_revenue

total_units_amazon = amazon_df["Units_Sold"].sum()
total_units_shopify = shopify_df["Units_Sold"].sum()
total_units = total_units_amazon + total_units_shopify

print("Total Amazon Revenue (GHS):", total_amazon_revenue)
print("Total Shopify Revenue (GHS):", total_shopify_revenue)
print("Combined Total Revenue (GHS):", total_revenue)
print("Total Units Sold:", total_units)


Total Amazon Revenue (GHS): 54480
Total Shopify Revenue (GHS): 36900
Combined Total Revenue (GHS): 91380
Total Units Sold: 768


In [9]:
# === TOP PRODUCTS BY REVENUE ===
top_amazon_products = amazon_df.groupby("Product")["Revenue_GHS"].sum().sort_values(ascending=False).head(5)
top_shopify_products = shopify_df.groupby("Product")["Revenue_GHS"].sum().sort_values(ascending=False).head(5)

print("Top 5 Amazon Products by Revenue:")
print(top_amazon_products, "\n")

print("Top 5 Shopify Products by Revenue:")
print(top_shopify_products)


Top 5 Amazon Products by Revenue:
Product
Wireless Mouse    18000
Laptop Stand      16200
Keyboard          14400
Phone Case         3000
Charger            2880
Name: Revenue_GHS, dtype: int64 

Top 5 Shopify Products by Revenue:
Product
T-Shirt     9000
Mug         8400
Hoodie      7500
Cap         7200
Tote Bag    4800
Name: Revenue_GHS, dtype: int64


In [11]:
region_sales = amazon_df.groupby("Region")["Revenue_GHS"].sum().sort_values(ascending=False)
print("Amazon Revenue by Region:")
print(region_sales)

Amazon Revenue by Region:
Region
Takoradi      18000
Kumasi        16200
Tamale        14400
Accra          3000
Cape Coast     2880
Name: Revenue_GHS, dtype: int64


In [12]:
region_sales = amazon_df.groupby("Region")["Revenue_GHS"].sum().sort_values(ascending=False)
print("Amazon Revenue by Region:")
print(region_sales)


Amazon Revenue by Region:
Region
Takoradi      18000
Kumasi        16200
Tamale        14400
Accra          3000
Cape Coast     2880
Name: Revenue_GHS, dtype: int64


In [14]:
# === DAILY SALES TREND ===
daily_sales = (
    amazon_df.groupby("Date")["Revenue_GHS"].sum() +
    shopify_df.groupby("Date")["Revenue_GHS"].sum()
).sort_index()

print("Daily Sales Trend:")
print(daily_sales)

Daily Sales Trend:
Date
2025-01-01    1750
2025-01-02    4100
2025-01-03    4200
2025-01-04    3900
2025-01-05    1280
2025-01-06    1750
2025-01-07    4100
2025-01-08    4200
2025-01-09    3900
2025-01-10    1280
2025-01-11    1750
2025-01-12    4100
2025-01-13    4200
2025-01-14    3900
2025-01-15    1280
2025-01-16    1750
2025-01-17    4100
2025-01-18    4200
2025-01-19    3900
2025-01-20    1280
2025-01-21    1750
2025-01-22    4100
2025-01-23    4200
2025-01-24    3900
2025-01-25    1280
2025-01-26    1750
2025-01-27    4100
2025-01-28    4200
2025-01-29    3900
2025-01-30    1280
Name: Revenue_GHS, dtype: int64


In [ ]:
# === CALCULATE ROI ===
ads_df["ROI_%"] = ((ads_df["Revenue_GHS"] - ads_df["Spend_GHS"]) / ads_df["Spend_GHS"]) * 100

# Average ROI by platform
roi_by_platform = ads_df.groupby("Platform")["ROI_%"].mean().sort_values(ascending=False)

print("Average ROI by Platform:")
print(roi_by_platform)

#This helps identify which ad platform (Meta, Google, TikTok, etc.) gives the best return.

Average ROI by Platform:
Platform
Meta      490.909091
Google    450.000000
TikTok    433.333333
Name: ROI_%, dtype: float64


In [16]:
# Combine Key Metrics into a Summary Report
summary = {
    "Total Amazon Revenue (GHS)": total_amazon_revenue,
    "Total Shopify Revenue (GHS)": total_shopify_revenue,
    "Total Combined Revenue (GHS)": total_revenue,
    "Total Units Sold": total_units,
    "Best Ad Platform (by ROI)": roi_by_platform.idxmax(),
    "Highest ROI (%)": round(roi_by_platform.max(), 2)
}

summary_df = pd.DataFrame(summary, index=[0])
print("Summary Report:")
print(summary_df)

Summary Report:
   Total Amazon Revenue (GHS)  Total Shopify Revenue (GHS)  \
0                       54480                        36900   

   Total Combined Revenue (GHS)  Total Units Sold Best Ad Platform (by ROI)  \
0                         91380               768                      Meta   

   Highest ROI (%)  
0           490.91  


In [ ]:
# Save KPI Summary for Power BI
summary_df.to_csv("kpi_summary.csv", index=False)